In [2]:
from datasets import load_dataset
import pandas as pd
import math

# Load the 'behavior' subset of the dataset
# Note: As observed previously, Hugging Face Hub might issue warnings for unauthenticated requests.
# It is recommended to log in to Hugging Face to enable higher rate limits and faster downloads.
# If you wish to do so, please let me know.
ds_behavior = load_dataset("liyucheng/goodreads", "behavior")

print("Dataset 'behavior' loaded successfully.")
print(f"Available splits: {list(ds_behavior.keys())}")

Loading dataset shards:   0%|          | 0/67 [00:00<?, ?it/s]

Dataset 'behavior' loaded successfully.
Available splits: ['train']


In [3]:
import pandas as pd
import gc
import os

# Required columns
required_columns = [
    'user_id',
    'book_id',
    'is_read',
    'rating',
    'date_added',
    'date_updated',
    'read_at',
    'started_at',
    'reading_duration_days'
]

train_split = ds_behavior['train']

# Keep only columns that actually exist
existing_columns = [
    col for col in required_columns
    if col in train_split.column_names
]

missing_columns = [
    col for col in required_columns
    if col not in train_split.column_names
]

if missing_columns:
    print("Missing columns:", missing_columns)

train_split = train_split.select_columns(existing_columns)

# Process exactly 30 million rows
max_rows = min(30_000_000, len(train_split))

# Process 1 million at a time
chunk_size = 1_000_000

# Output files
read_file = "read_interactions.csv"
unread_file = "unread_interactions.csv"

# Remove old files if they already exist
if os.path.exists(read_file):
    os.remove(read_file)

if os.path.exists(unread_file):
    os.remove(unread_file)

total_read = 0
total_unread = 0

print(f"Processing {max_rows:,} rows...")
print("-" * 60)

for start in range(0, max_rows, chunk_size):

    end = min(start + chunk_size, max_rows)

    print(f"Processing rows {start:,} - {end:,}")

    # Get only this chunk
    chunk_dataset = train_split.select(range(start, end))

    # Convert ONLY this chunk to pandas
    df_chunk = chunk_dataset.to_pandas()

    # -----------------------------
    # Read interactions
    # -----------------------------
    df_read_chunk = df_chunk[
        (df_chunk['is_read'] == 1) &
        (df_chunk['reading_duration_days'].notna())
    ]

    # -----------------------------
    # Unread interactions
    # -----------------------------
    df_unread_chunk = df_chunk[
        df_chunk['is_read'] == 0
    ]

    # Append directly to CSV
    if len(df_read_chunk) > 0:
        df_read_chunk.to_csv(
            read_file,
            mode='a',
            header=not os.path.exists(read_file),
            index=False
        )

    if len(df_unread_chunk) > 0:
        df_unread_chunk.to_csv(
            unread_file,
            mode='a',
            header=not os.path.exists(unread_file),
            index=False
        )

    total_read += len(df_read_chunk)
    total_unread += len(df_unread_chunk)

    print(
        f"Completed: {end:,}/{max_rows:,} | "
        f"Read: {total_read:,} | "
        f"Unread: {total_unread:,}"
    )

    # VERY IMPORTANT: release memory
    del chunk_dataset
    del df_chunk
    del df_read_chunk
    del df_unread_chunk

    gc.collect()

print("-" * 60)
print("Processing completed!")

print(f"Total read interactions:   {total_read:,}")
print(f"Total unread interactions: {total_unread:,}")

print(f"\nCreated:")
print(read_file)
print(unread_file)

print("\nFile sizes:")
print(f"Read:   {os.path.getsize(read_file) / (1024**3):.2f} GB")
print(f"Unread: {os.path.getsize(unread_file) / (1024**3):.2f} GB")

Processing 30,000,000 rows...
------------------------------------------------------------
Processing rows 0 - 1,000,000
Completed: 1,000,000/30,000,000 | Read: 137,969 | Unread: 517,662
Processing rows 1,000,000 - 2,000,000
Completed: 2,000,000/30,000,000 | Read: 273,757 | Unread: 1,039,132
Processing rows 2,000,000 - 3,000,000
Completed: 3,000,000/30,000,000 | Read: 413,380 | Unread: 1,515,537
Processing rows 3,000,000 - 4,000,000
Completed: 4,000,000/30,000,000 | Read: 533,567 | Unread: 1,981,676
Processing rows 4,000,000 - 5,000,000
Completed: 5,000,000/30,000,000 | Read: 666,112 | Unread: 2,492,727
Processing rows 5,000,000 - 6,000,000
Completed: 6,000,000/30,000,000 | Read: 814,265 | Unread: 3,020,609
Processing rows 6,000,000 - 7,000,000
Completed: 7,000,000/30,000,000 | Read: 964,502 | Unread: 3,548,013
Processing rows 7,000,000 - 8,000,000
Completed: 8,000,000/30,000,000 | Read: 1,089,250 | Unread: 4,068,117
Processing rows 8,000,000 - 9,000,000
Completed: 9,000,000/30,000,000

In [5]:
import pandas as pd
import gc
from google.colab import files

# --- Step 1: Load Data ---
print("Loading read_interactions.csv...")
df_read = pd.read_csv('read_interactions.csv')
print(f"Read interactions loaded: {len(df_read):,} rows")

print("Loading unread_interactions.csv...")
df_unread = pd.read_csv('unread_interactions.csv')
print(f"Unread interactions loaded: {len(df_unread):,} rows")

# --- Step 2: Combine Data ---
print("Combining read and unread interactions...")
df_combined = pd.concat([df_read, df_unread], ignore_index=True)
print(f"Combined dataset size: {len(df_combined):,} rows")

del df_read, df_unread
gc.collect()

# --- Step 3: Initial User Interaction Filter ---
# Make sure each user_id has at least >=10 interactions in the entire combined dataset
print("Filtering users with less than 10 total interactions...")
user_counts = df_combined['user_id'].value_counts()
valid_users_initial = user_counts[user_counts >= 10].index
df_filtered_initial = df_combined[df_combined['user_id'].isin(valid_users_initial)]
print(f"Combined dataset after initial user filter: {len(df_filtered_initial):,} rows from {len(valid_users_initial):,} unique users")

del df_combined, user_counts, valid_users_initial
gc.collect()

# --- Step 4: Balance Read/Unread Interactions ---
print("Balancing read and unread interactions...")
df_read_temp = df_filtered_initial[df_filtered_initial['is_read'] == 1]
df_unread_temp = df_filtered_initial[df_filtered_initial['is_read'] == 0]

min_class_size = min(len(df_read_temp), len(df_unread_temp))

print(f"Number of read interactions for balancing: {len(df_read_temp):,}")
print(f"Number of unread interactions for balancing: {len(df_unread_temp):,}")
print(f"Balancing to {min_class_size:,} entries for each class.")

df_read_balanced = df_read_temp.sample(n=min_class_size, random_state=42)
df_unread_balanced = df_unread_temp.sample(n=min_class_size, random_state=42)

df_final_balanced = pd.concat([df_read_balanced, df_unread_balanced], ignore_index=True)
print(f"Dataset after balancing: {len(df_final_balanced):,} rows")

del df_read_temp, df_unread_temp, df_read_balanced, df_unread_balanced, df_filtered_initial
gc.collect()

# --- Step 5: Final User Interaction Check and Filter (to guarantee the condition on the final output) ---
print("Re-checking user interaction counts after balancing...")
final_user_counts = df_final_balanced['user_id'].value_counts()
final_valid_users = final_user_counts[final_user_counts >= 10].index
df_final_balanced_and_filtered = df_final_balanced[df_final_balanced['user_id'].isin(final_valid_users)]
print(f"Final balanced dataset after re-filtering for >=10 interactions per user: {len(df_final_balanced_and_filtered):,} rows from {len(final_valid_users):,} unique users")

del df_final_balanced, final_user_counts, final_valid_users
gc.collect()

# --- Step 6: Save to CSV ---
output_filename = 'balanced_filtered_interactions.csv'
print(f"Saving final balanced and filtered dataset to '{output_filename}'...")
df_final_balanced_and_filtered.to_csv(output_filename, index=False)
print(f"'{output_filename}' created successfully.")

# --- Download the file ---
print(f"Downloading '{output_filename}'...")
files.download(output_filename)
print(f"'{output_filename}' has been downloaded.")

Loading read_interactions.csv...
Read interactions loaded: 4,213,336 rows
Loading unread_interactions.csv...
Unread interactions loaded: 15,311,135 rows
Combining read and unread interactions...
Combined dataset size: 19,524,471 rows
Filtering users with less than 10 total interactions...
Combined dataset after initial user filter: 19,427,227 rows from 86,004 unique users
Balancing read and unread interactions...
Number of read interactions for balancing: 4,195,669
Number of unread interactions for balancing: 15,231,558
Balancing to 4,195,669 entries for each class.
Dataset after balancing: 8,391,338 rows
Re-checking user interaction counts after balancing...
Final balanced dataset after re-filtering for >=10 interactions per user: 8,297,731 rows from 69,475 unique users
Saving final balanced and filtered dataset to 'balanced_filtered_interactions.csv'...
'balanced_filtered_interactions.csv' created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'balanced_filtered_interactions.csv' has been downloaded.


In [6]:
import pandas as pd

# Load the balanced_filtered_interactions.csv file
output_filename = 'balanced_filtered_interactions.csv'
try:
    df_balanced = pd.read_csv(output_filename)
    print(f"Loaded '{output_filename}'. Total rows: {len(df_balanced):,}")

    # Get the count of 'read' (is_read == 1) and 'unread' (is_read == 0) interactions
    read_count = df_balanced[df_balanced['is_read'] == 1].shape[0]
    unread_count = df_balanced[df_balanced['is_read'] == 0].shape[0]

    print(f"\nTotal 'read' interactions: {read_count:,}")
    print(f"Total 'unread' interactions: {unread_count:,}")
    print(f"Sum of counts: {read_count + unread_count:,} (should match total rows)")

except FileNotFoundError:
    print(f"Error: The file '{output_filename}' was not found. Please ensure the previous steps to create and save this file were executed successfully.")
except Exception as e:
    print(f"An error occurred: {e}")

/tmp/ipykernel_5809/706966450.py:6: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_balanced = pd.read_csv(output_filename)


Loaded 'balanced_filtered_interactions.csv'. Total rows: 8,297,731

Total 'read' interactions: 4,170,967
Total 'unread' interactions: 4,126,764
Sum of counts: 8,297,731 (should match total rows)


In [1]:
import pandas as pd
import gc

# Define the target size for each class
TARGET_CLASS_SIZE = 100000
MIN_INTERACTIONS_PER_USER = 10

output_filename = 'balanced_filtered_interactions.csv'

try:
    # Load the balanced_filtered_interactions.csv file
    df_balanced = pd.read_csv(output_filename)
    print(f"Loaded '{output_filename}'. Total rows: {len(df_balanced):,}")

    # Step 1: Separate read and unread interactions
    df_read = df_balanced[df_balanced['is_read'] == 1]
    df_unread = df_balanced[df_balanced['is_read'] == 0]

    print(f"Original read interactions: {len(df_read):,}")
    print(f"Original unread interactions: {len(df_unread):,}")

    # Step 2: Sample 100k read and 100k unread interactions
    if len(df_read) >= TARGET_CLASS_SIZE and len(df_unread) >= TARGET_CLASS_SIZE:
        df_read_sampled = df_read.sample(n=TARGET_CLASS_SIZE, random_state=42)
        df_unread_sampled = df_unread.sample(n=TARGET_CLASS_SIZE, random_state=42)

        df_200k_dataset = pd.concat([df_read_sampled, df_unread_sampled], ignore_index=True)
        print(f"Created 200k dataset with {len(df_read_sampled):,} read and {len(df_unread_sampled):,} unread interactions.")

        del df_read, df_unread, df_read_sampled, df_unread_sampled, df_balanced
        gc.collect()

        # Step 3: Filter for users with at least 10 interactions in the NEW 200k dataset
        print(f"Re-checking user interaction counts in the 200k dataset...")
        user_counts_200k = df_200k_dataset['user_id'].value_counts()
        valid_users_200k = user_counts_200k[user_counts_200k >= MIN_INTERACTIONS_PER_USER].index

        df_final_200k_filtered = df_200k_dataset[df_200k_dataset['user_id'].isin(valid_users_200k)]

        print(f"Final 200k dataset after filtering for >=10 interactions per user: {len(df_final_200k_filtered):,} rows from {len(valid_users_200k):,} unique users.")

        # Step 4: Calculate and print user interaction summary for the final 200k dataset
        user_summary_200k = df_final_200k_filtered.groupby('user_id').agg(
            total_entries=('book_id', 'count'),
            read_entries=('is_read', lambda x: (x == 1).sum()),
            unread_entries=('is_read', lambda x: (x == 0).sum())
        ).reset_index()

        # Verify balancing in the final dataset
        final_read_count = df_final_200k_filtered[df_final_200k_filtered['is_read'] == 1].shape[0]
        final_unread_count = df_final_200k_filtered[df_final_200k_filtered['is_read'] == 0].shape[0]
        print(f"\nFinal dataset contains {final_read_count:,} read interactions and {final_unread_count:,} unread interactions.")

        print(f"\nInteraction summary for users in the final 200k dataset (first 10 users):")
        print(user_summary_200k.head(10).to_markdown(index=False))

        print(f"\nTotal unique users in this summary: {len(user_summary_200k):,}")
        print("\nFull summary can be viewed by inspecting 'user_summary_200k' variable.")

    else:
        print(f"Error: Not enough read ({len(df_read):,}) or unread ({len(df_unread):,}) interactions to sample {TARGET_CLASS_SIZE:,} for each class.")

except FileNotFoundError:
    print(f"Error: The file '{output_filename}' was not found. Please ensure the previous steps to create and save this file were executed successfully.")
except Exception as e:
    print(f"An error occurred: {e}")

Loaded 'read_interactions.csv'. Total rows: 4,213,336
Loaded 'unread_interactions.csv'. Total rows: 15,311,135
Created 200k dataset with 100,000 read and 100,000 unread interactions.
Final 200k dataset (without re-filtering for >=10 interactions per user): 200,000 rows from 49,359 unique users.

Final dataset contains 100,000 read interactions and 100,000 unread interactions.

Interaction summary for users in the final 200k dataset (first 10 users):
| user_id                          |   total_entries |   read_entries |   unread_entries |
|:---------------------------------|----------------:|---------------:|-----------------:|
| 0003f216823de684cea464170efe1d42 |               1 |              0 |                1 |
| 0006f3daa8a5ecd94788d9f08658754f |               1 |              0 |                1 |
| 00081491f89dd0e43b57f7ad39ca97a0 |               7 |              0 |                7 |
| 000a9539c044b397780166167b1327b3 |               5 |              5 |                0 |


In [3]:
import pandas as pd
from google.colab import files

# Define the output filename for the new 200k dataset
output_200k_filename = 'final_200k_filtered_interactions.csv'

# Ensure df_final_200k_filtered exists from the previous cell's execution
if 'df_final_200k_filtered' in locals() and not df_final_200k_filtered.empty:
    print(f"Saving the final 200k dataset to '{output_200k_filename}'...")
    df_final_200k_filtered.to_csv(output_200k_filename, index=False)
    print(f"'{output_200k_filename}' created successfully.")

    # Download the file
    print(f"Downloading '{output_200k_filename}'...")
    files.download(output_200k_filename)
    print(f"'{output_200k_filename}' has been downloaded.")
else:
    print("Error: The DataFrame 'df_final_200k_filtered' was not found or is empty. Please ensure the previous step to create this dataset was executed successfully.")

Saving the final 200k dataset to 'final_200k_filtered_interactions.csv'...
'final_200k_filtered_interactions.csv' created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'final_200k_filtered_interactions.csv' has been downloaded.


In [4]:
import pandas as pd

# Define the filename of the final 200k dataset
filename = 'final_200k_filtered_interactions.csv'

try:
    # Load the dataset
    df_final_200k = pd.read_csv(filename)
    print(f"Loaded '{filename}'. Total rows: {len(df_final_200k):,}")

    # Ensure 'date_added' column is in datetime format
    if 'date_added' in df_final_200k.columns:
        df_final_200k['date_added'] = pd.to_datetime(df_final_200k['date_added'], errors='coerce')

        # Filter out rows where date_added could not be parsed (NaT values)
        df_final_200k = df_final_200k.dropna(subset=['date_added'])

        if not df_final_200k.empty:
            # Get minimum and maximum date_added
            min_date_added = df_final_200k['date_added'].min()
            max_date_added = df_final_200k['date_added'].max()

            print(f"\nMinimum 'date_added': {min_date_added}")
            print(f"Maximum 'date_added': {max_date_added}")
        else:
            print("No valid 'date_added' entries found after parsing.")
    else:
        print(f"Error: 'date_added' column not found in '{filename}'.")

except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found. Please ensure it has been created and saved.")
except Exception as e:
    print(f"An error occurred: {e}")

Loaded 'final_200k_filtered_interactions.csv'. Total rows: 82,855

Minimum 'date_added': 2004-04-14 00:00:00
Maximum 'date_added': 2017-11-03 19:53:13


/tmp/ipykernel_15899/2261211374.py:8: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final_200k = pd.read_csv(filename)


In [7]:
import pandas as pd
from datasets import load_dataset
import gc

# --- Step 1: Load the final 200k interactions dataset ---
interactions_filename = 'final_200k_filtered_interactions.csv'
try:
    df_interactions = pd.read_csv(interactions_filename)
    print(f"Loaded '{interactions_filename}'. Total rows: {len(df_interactions):,}")
except FileNotFoundError:
    print(f"Error: The file '{interactions_filename}' was not found. Please ensure it has been created and saved.")
    exit()
except Exception as e:
    print(f"An error occurred while loading interactions data: {e}")
    exit()

# --- Step 2: Load the 'books' dataset ---
print("Loading 'books' subset from 'liyucheng/goodreads' dataset...")
try:
    # Login using e.g. `huggingface-cli login` to access this dataset
    # Note: If you encounter authentication issues or rate limits, consider logging in.
    ds_books = load_dataset("liyucheng/goodreads", "books")
    print("Dataset 'books' loaded successfully.")
except Exception as e:
    print(f"An error occurred while loading books dataset: {e}")
    exit()

# Convert to pandas DataFrame for easier merging
# Selecting only necessary columns to reduce memory usage
book_columns_to_keep = [
    'book_id',
    'title',
    'author_ids',
    'author_names',
    'average_rating',
    'publication_year',
    'description',
    'genres',
    'num_pages',
    'original_title',
    'original_publication_year'
]

# Convert Dataset object to pandas DataFrame for the 'train' split (or just select relevant columns)
# The 'books' dataset usually has only one split, often named 'train' or implicitly accessible.
# We'll try to convert to pandas, assuming it's structured for direct conversion.
# Check available splits and take the first one, or directly access if there's no split name like 'train'
if 'train' in ds_books:
    df_books = ds_books['train'].to_pandas()
else:
    # Fallback for datasets without named splits, assuming direct access is possible
    df_books = pd.DataFrame(ds_books)

# Filter df_books to keep only the specified columns
df_books_filtered = df_books[book_columns_to_keep]
print(f"Books data loaded with {len(df_books_filtered):,} entries and {len(book_columns_to_keep)} columns.")

del ds_books, df_books
gc.collect()

# --- Step 3: Merge the datasets ---
print("Merging interactions dataset with book data on 'book_id'...")
# Ensure 'book_id' is of the same type for merging
df_interactions['book_id'] = df_interactions['book_id'].astype(str)
df_books_filtered['book_id'] = df_books_filtered['book_id'].astype(str)

df_merged = pd.merge(df_interactions, df_books_filtered, on='book_id', how='left')
print(f"Merged dataset created. Total rows: {len(df_merged):,}")
print(f"Total columns in merged dataset: {len(df_merged.columns)}")

del df_interactions, df_books_filtered
gc.collect()

# --- Step 4: Display results ---
print("\nSample of the merged dataset (first 5 rows with new book columns):")
# Displaying interaction columns and the newly added book columns
columns_to_display = [
    'user_id', 'book_id', 'is_read', 'rating', 'date_added',
    'title', 'author_names', 'average_rating', 'genres', 'description'
]
# Ensure columns actually exist before trying to display them
existing_cols = [col for col in columns_to_display if col in df_merged.columns]
print(df_merged[existing_cols].head().to_markdown(index=False))

print("\nFull merged dataset is stored in the 'df_merged' DataFrame.")

Loaded 'final_200k_filtered_interactions.csv'. Total rows: 82,855
Loading 'books' subset from 'liyucheng/goodreads' dataset...


KeyboardInterrupt: 

In [6]:
df_merged.head()

,user_id,book_id,is_read,rating,date_added,date_updated,read_at,started_at,reading_duration_days,title,author_ids,author_names,average_rating,publication_year,description,genres,num_pages,original_title,original_publication_year
0,6ae8a47b5a7f8b0b145cf597a5d5c929,17342700,1,5,2016-12-21 11:23:08,2017-04-24 19:20:25,2017-04-24 00:00:00,2017-04-22 00:00:00,2.0,"Gathering Darkness (Falling Kingdoms, #3)","[5295120, 38531]","[Morgan Rhodes, Michelle Rowen]",4.32,2014,"In GATHERING DARKNESS, book three of the New Y...","[to-read, young-adult, currently-reading, fant...",416,Gathering Darkness,2014
1,0c7429f150417a1af3c7045fc89a3f4e,18513581,1,3,2016-07-23 07:06:57,2017-01-01 17:40:35,2016-12-10 00:00:00,2016-12-02 00:00:00,8.0,"Young Avengers, Volume 2: Alternative Culture","[439887, 221821, 504788, 93125]","[Kieron Gillen, Jamie McKelvie, Kate Brown, Ma...",4.18,2014,Ever wonder what the super hero equivalent of ...,"[to-read, comics, graphic-novels, marvel, grap...",112,"Young Avengers, Volume 2: Alternative Culture",2014
2,cd75b15c85a3dd27fef5ec3403a5f873,3993755,1,0,2017-06-30 08:33:08,2017-06-30 08:33:26,2017-06-30 08:33:26,2017-06-29 00:00:00,1.0,"Hail to the Chef (A White House Chef Mystery, #2)",[1020232],[Julie Hyzy],3.98,2008,White House executive chef Ollie Paras has to ...,"[to-read, mystery, cozy-mystery, cozy-mysterie...",328,"Hail to the Chef (White House Chef Mystery, Bo...",2008
3,86b931f21144e8d743b40dc26f82073c,89724,1,4,2011-10-07 13:43:08,2011-11-03 20:05:41,2011-10-13 00:00:00,2011-10-07 00:00:00,6.0,We Have Always Lived in the Castle,"[13388, 6404]","[Shirley Jackson, Jonathan Lethem]",4.04,2006,Merricat Blackwood lives on the family estate ...,"[to-read, horror, fiction, favorites, currentl...",146,We Have Always Lived in the Castle,1962
4,9adddbae754bf26fb04999a0f79f14c4,13565914,1,3,2013-01-03 18:42:59,2013-01-03 18:51:59,2013-01-04 18:51:59,2013-01-01 00:00:00,3.0,"Knight (Unfinished Hero, #1)",[2958084],[Kristen Ashley],3.94,2012,Anya Gage has learned that to get anything goo...,"[to-read, favorites, romance, currently-readin...",285,Knight,2012


In [8]:
import pandas as pd
from google.colab import files

# Define the output filename for the merged dataset
output_merged_filename = 'merged_interactions_with_books.csv'

# Ensure df_merged exists
if 'df_merged' in locals() and not df_merged.empty:
    print(f"Saving the merged dataset to '{output_merged_filename}'...")
    df_merged.to_csv(output_merged_filename, index=False)
    print(f"'{output_merged_filename}' created successfully.")

    # Download the file
    print(f"Downloading '{output_merged_filename}'...")
    files.download(output_merged_filename)
    print(f"'{output_merged_filename}' has been downloaded.")
else:
    print("Error: The DataFrame 'df_merged' was not found or is empty. Please ensure the merging step was executed successfully.")

Saving the merged dataset to 'merged_interactions_with_books.csv'...
'merged_interactions_with_books.csv' created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'merged_interactions_with_books.csv' has been downloaded.


In [9]:
import pandas as pd

# Ensure df_merged is available
if 'df_merged' not in locals():
    print("Error: 'df_merged' DataFrame not found. Please ensure the previous merging step was executed successfully.")
else:
    print("Analyzing 'date_added' and 'read_at' from 'df_merged'...")

    # Convert 'date_added' and 'read_at' to datetime objects
    # Coercing errors will turn unparseable dates into NaT (Not a Time)
    df_merged['date_added'] = pd.to_datetime(df_merged['date_added'], errors='coerce')
    df_merged['read_at'] = pd.to_datetime(df_merged['read_at'], errors='coerce')

    # Drop rows where 'date_added' or 'read_at' could not be parsed
    df_analysis = df_merged.dropna(subset=['date_added', 'read_at']).copy()

    if df_analysis.empty:
        print("No valid 'date_added' and 'read_at' entries found after parsing and dropping NaNs.")
    else:
        # 1. Min, Max, Average, Median of date_added
        min_date_added = df_analysis['date_added'].min()
        max_date_added = df_analysis['date_added'].max()
        avg_date_added = df_analysis['date_added'].mean()
        median_date_added = df_analysis['date_added'].median()

        print("\n--- 'date_added' Statistics ---")
        print(f"Minimum 'date_added': {min_date_added}")
        print(f"Maximum 'date_added': {max_date_added}")
        print(f"Average 'date_added': {avg_date_added}")
        print(f"Median 'date_added': {median_date_added}")

        # 2. Calculate the difference between 'read_at' and 'date_added'
        df_analysis['read_duration'] = df_analysis['read_at'] - df_analysis['date_added']

        # Convert timedelta to days
        df_analysis['read_duration_days'] = df_analysis['read_duration'].dt.days

        # Filter out negative durations (where read_at is before date_added, which could be data errors)
        df_valid_duration = df_analysis[df_analysis['read_duration_days'] >= 0]
        if len(df_analysis) - len(df_valid_duration) > 0:
            print(f"\nWarning: {len(df_analysis) - len(df_valid_duration):,} entries had 'read_at' before 'date_added' and were excluded from duration analysis.")

        # Count interactions based on duration
        less_than_eq_90_days = df_valid_duration[df_valid_duration['read_duration_days'] <= 90].shape[0]
        greater_than_90_days = df_valid_duration[df_valid_duration['read_duration_days'] > 90].shape[0]

        print("\n--- 'read_at' vs 'date_added' Duration Analysis ---")
        print(f"Number of interactions where (read_at - date_added) <= 90 days: {less_than_eq_90_days:,}")
        print(f"Number of interactions where (read_at - date_added) > 90 days: {greater_than_90_days:,}")

Analyzing 'date_added' and 'read_at' from 'df_merged'...

--- 'date_added' Statistics ---
Minimum 'date_added': 2004-04-14 00:00:00
Maximum 'date_added': 2017-10-27 18:20:28
Average 'date_added': 2014-10-15 12:53:14.909506816
Median 'date_added': 2015-01-26 23:24:15


--- 'read_at' vs 'date_added' Duration Analysis ---
Number of interactions where (read_at - date_added) <= 90 days: 26,822
Number of interactions where (read_at - date_added) > 90 days: 9,492


In [10]:
if 'df_merged' in locals():
    unique_user_ids_count = df_merged['user_id'].nunique()
    print(f"Number of unique user IDs in the merged dataset: {unique_user_ids_count:,}")
else:
    print("Error: 'df_merged' DataFrame not found. Please ensure the merging step was executed successfully.")

Number of unique user IDs in the merged dataset: 4,336


In [3]:
import pandas as pd

# --- Load df_merged from CSV as requested by the user ---
merged_filename = 'merged_interactions_with_books.csv'
try:
    df_merged = pd.read_csv(merged_filename)
    print(f"Loaded '{merged_filename}'. Total rows: {len(df_merged):,}")
except FileNotFoundError:
    print(f"Error: The file '{merged_filename}' was not found. Please ensure the merging step was executed successfully.")
    # Exit or handle error gracefully if the file is crucial
    exit()
except Exception as e:
    print(f"An error occurred while loading merged data: {e}")
    exit()

# Ensure df_merged is available
if df_merged.empty:
    print("Error: 'df_merged' DataFrame is empty after loading. Please check the content of the CSV.")
else:
    # Create a copy to work on, avoiding SettingWithCopyWarning and keeping original df_merged untouched until confirmed
    df_with_target = df_merged.copy()

    # Step 1: Ensure date columns are in datetime format
    # Coercing errors will turn unparseable dates into NaT (Not a Time)
    df_with_target['date_added'] = pd.to_datetime(df_with_target['date_added'], errors='coerce')
    df_with_target['read_at'] = pd.to_datetime(df_with_target['read_at'], errors='coerce')

    # Step 2: Initialize 'target' column with NaN (representing EXCLUDE state or undefined)
    # Rows that don't meet any of the assignment criteria will remain NA (effectively EXCLUDE)
    df_with_target['target'] = pd.NA

    # Define the date threshold from the user's diagram
    threshold_date = pd.to_datetime('2017-03-12')

    # Identify rows that are eligible for target assignment based on 'date_added'
    # These are rows where date_added is valid and <= the threshold date.
    eligible_for_processing_mask = (df_with_target['date_added'].notna()) & \
                                   (df_with_target['date_added'] <= threshold_date)

    # --- Apply target = 0 for UNREAD books within the eligible window ---
    # Condition: (eligible_for_processing_mask) AND (df_with_target['is_read'] == 0)
    cond_unread_target_0 = eligible_for_processing_mask & (df_with_target['is_read'] == 0)
    df_with_target.loc[cond_unread_target_0, 'target'] = 0

    # --- Apply target for READ books within the eligible window ---
    # Condition: (eligible_for_processing_mask) AND (df_with_target['is_read'] == 1) AND (read_at is not NaT)
    cond_read_eligible_mask = eligible_for_processing_mask & \
                              (df_with_target['is_read'] == 1) & \
                              (df_with_target['read_at'].notna())

    # Calculate read duration in days for these specific rows
    # A temporary column is created for this calculation.
    df_with_target.loc[cond_read_eligible_mask, 'read_duration_days'] = \
        (df_with_target.loc[cond_read_eligible_mask, 'read_at'] - \
         df_with_target.loc[cond_read_eligible_mask, 'date_added']).dt.days

    # Filter for valid read durations (non-negative days). Negative durations are treated as 'EXCLUDE' (remain NA).
    cond_valid_duration_mask = cond_read_eligible_mask & (df_with_target['read_duration_days'] >= 0)

    # Sub-case: read_duration <= 90 days (target = 1)
    cond_read_target_1 = cond_valid_duration_mask & (df_with_target['read_duration_days'] <= 90)
    df_with_target.loc[cond_read_target_1, 'target'] = 1

    # Sub-case: read_duration > 90 days (target = 0)
    cond_read_target_0 = cond_valid_duration_mask & (df_with_target['read_duration_days'] > 90)
    df_with_target.loc[cond_read_target_0, 'target'] = 0

    # Drop the temporary 'read_duration_days' column if it was created
    df_with_target = df_with_target.drop(columns=['read_duration_days'], errors='ignore')

    # Replace the original df_merged with the new DataFrame containing the 'target' column
    df_merged = df_with_target
    print(f"Added 'target' column to 'df_merged'. Total rows: {len(df_merged):,}")

    # Display value counts for the new 'target' column
    print("\nValue counts for the 'target' column:")
    # Print value_counts directly to avoid TypeError with to_markdown() and pd.NA
    print(df_merged['target'].value_counts(dropna=False))

    # Display a sample of rows with the new 'target' column
    print("\nSample of df_merged with new 'target' column (first 5 rows):")
    print(df_merged[['user_id', 'book_id', 'is_read', 'date_added', 'read_at', 'target']].head().to_markdown(index=False))

    # Also show some rows with NA target values (excluded) for verification
    na_target_rows = df_merged[df_merged['target'].isna()]
    if not na_target_rows.empty:
        print("\nSample of rows where 'target' is NA (excluded) (first 5 rows):")
        print(na_target_rows[['user_id', 'book_id', 'is_read', 'date_added', 'read_at', 'target']].head().to_markdown(index=False))
    else:
        print("\nNo rows found with 'target' as NA (all rows were assigned a target value).")


/tmp/ipykernel_56917/32643217.py:6: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_merged = pd.read_csv(merged_filename)


Loaded 'merged_interactions_with_books.csv'. Total rows: 82,855
Added 'target' column to 'df_merged'. Total rows: 82,855

Value counts for the 'target' column:
target
0       47726
1       23661
<NA>    11468
Name: count, dtype: int64

Sample of df_merged with new 'target' column (first 5 rows):
| user_id                          |   book_id |   is_read | date_added          | read_at             | target   |
|:---------------------------------|----------:|----------:|:--------------------|:--------------------|:---------|
| 6ae8a47b5a7f8b0b145cf597a5d5c929 |  17342700 |         1 | 2016-12-21 11:23:08 | 2017-04-24 00:00:00 | 0        |
| 0c7429f150417a1af3c7045fc89a3f4e |  18513581 |         1 | 2016-07-23 07:06:57 | 2016-12-10 00:00:00 | 0        |
| cd75b15c85a3dd27fef5ec3403a5f873 |   3993755 |         1 | 2017-06-30 08:33:08 | 2017-06-30 08:33:26 | <NA>     |
| 86b931f21144e8d743b40dc26f82073c |     89724 |         1 | 2011-10-07 13:43:08 | 2011-10-13 00:00:00 | 1        |
| 9addd

In [4]:
df_merged.shape

(82855, 20)

In [5]:
import pandas as pd
from google.colab import files

# Define the output filename for the final dataset including the target column
output_final_filename = 'final_merged_data_with_target.csv'

# Ensure df_merged is available and contains the target column
if 'df_merged' in locals() and not df_merged.empty and 'target' in df_merged.columns:
    print(f"Saving the final dataset to '{output_final_filename}'...")
    df_merged.to_csv(output_final_filename, index=False)
    print(f"'{output_final_filename}' created successfully.")

    # Download the file
    print(f"Downloading '{output_final_filename}'...")
    files.download(output_final_filename)
    print(f"'{output_final_filename}' has been downloaded.")
elif 'df_merged' not in locals() or df_merged.empty:
    print("Error: The DataFrame 'df_merged' was not found or is empty. Please ensure all previous steps were executed successfully.")
else:
    print("Error: The 'target' column was not found in 'df_merged'. Please ensure the target column creation step was executed correctly.")


Saving the final dataset to 'final_merged_data_with_target.csv'...
'final_merged_data_with_target.csv' created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'final_merged_data_with_target.csv' has been downloaded.
